# LLM08: Mixture of Experts (MoE) & Numerical Precision

## Lab Overview

1. **Mixture of Experts (MoE)**: A sparsely-activated architecture that scales model capacity without proportionally increasing compute. Models like Mixtral 8×7B, DeepSeek-V3 (256+1 experts, top-8+1), and Qwen3-235B (128 experts, top-8) use MoE to activate only a subset of parameters per token.
2. **Numerical Precision & Quantization**: Understanding FP32, BF16, FP16, FP8 representations and how quantization (INT8/INT4) reduces model size and speeds up inference.

> **Quick term:** **MoE (Mixture of Experts)** means many separate “expert” feed-forward networks plus a **router** that picks which expert(s) run for each token—only a few experts activate, so compute stays manageable while capacity grows.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.

## Goals

By the end of this lab, you will be able to:

1. Understand the MoE architecture: expert networks, gating mechanism, and sparse activation.
2. Implement Top-K gating and expert routing from scratch.
3. Explore load balancing strategies for MoE training.
4. Compare floating-point formats (FP32, FP16, BF16, FP8) and their trade-offs.
5. Implement basic quantization (INT8/INT4) and measure its impact on model accuracy.

---


## 1. Environment Setup

In [2]:
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

Using device: cuda
PyTorch version: 2.6.0+gitdbfe118
GPU: AMD Radeon Graphics
GPU Memory: 12.4 GB


## 2. MoE Concept

The Mixture of Experts idea is simple: instead of one large FFN, use **multiple smaller FFNs** ("experts") and a **gating network** ("router") that decides which expert(s) to activate for each input token.

**Key equation:**
$$y = \sum_{i=1}^{M} G(x)_i \cdot E_i(x)$$

where $G(x)$ is the gating function and $E_i$ is expert $i$.

**Sparsity via Top-K:**
$$G(x) = \text{softmax}(\text{TopK}(x \cdot W_g))$$

Only $k$ experts (typically $k=1$ or $2$) are activated per token. This means:
- **Model capacity** scales with $M$ (number of experts)
- **Compute cost** stays roughly at $k \times$ single-expert cost

**Notable MoE models:**

| Model | #Experts | Top-K | Notes |
|-------|----------|-------|-------|
| Mixtral 8×7B | 8 | 2 | Popular open-source MoE |
| Mixtral 8×22B | 8 | 2 | Larger experts |
| DeepSeek-V3 (670B/37B active) | 256+1 | 8+1 | Shared expert + routing |
| Qwen3-235B-A22B | 128 | 8 | Alibaba open-source |

In [3]:
# Step 1: Implement the Top-K Gating Network

class TopKGating(nn.Module):
    """Router that selects top-k experts for each token."""

    def __init__(self, hidden_dim: int, num_experts: int, k: int = 2):
        super().__init__()
        self.router = nn.Linear(hidden_dim, num_experts, bias=False)
        self.k = k
        self.num_experts = num_experts

    def forward(self, x: torch.Tensor):
        """
        Args:
            x: (batch, seq_len, hidden_dim)
        Returns:
            gate_probs: (batch, seq_len, num_experts) — sparse, only top-k nonzero
            topk_idx:   (batch, seq_len, k) — indices of selected experts
        """
        scores = self.router(x)                                     # (B, S, E)
        topk_scores, topk_idx = torch.topk(scores, self.k, dim=-1) # (B, S, k)
        probs = torch.softmax(topk_scores, dim=-1)                 # normalize among selected

        # Build sparse gate: only top-k positions are nonzero
        gate = torch.zeros_like(scores)
        gate.scatter_(-1, topk_idx, probs)

        return gate, topk_idx


# Test the router
hidden_dim, num_experts, k = 64, 8, 2
gating = TopKGating(hidden_dim, num_experts, k).to(device)

x = torch.randn(2, 5, hidden_dim, device=device)  # batch=2, seq_len=5
gate, idx = gating(x)

print(f"Input shape: {x.shape}")
print(f"Gate shape:  {gate.shape}  (sparse — only {k} nonzero per token)")
print(f"Top-K idx:   {idx.shape}")
print(f"\nSample gate[0,0]: {gate[0, 0].detach().cpu().tolist()}")
print(f"Selected experts for token [0,0]: {idx[0, 0].tolist()}")
print(f"Gate sums to 1: {gate[0, 0].sum().item():.4f}")

Input shape: torch.Size([2, 5, 64])
Gate shape:  torch.Size([2, 5, 8])  (sparse — only 2 nonzero per token)
Top-K idx:   torch.Size([2, 5, 2])

Sample gate[0,0]: [0.4754689037799835, 0.5245311260223389, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Selected experts for token [0,0]: [1, 0]
Gate sums to 1: 1.0000


In [4]:
# Step 2: Implement individual Expert (standard FFN with SiLU gating, LLaMA-style)

class Expert(nn.Module):
    """A single FFN expert — identical structure to LLaMA's MLP."""

    def __init__(self, hidden_dim: int, intermediate_dim: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.up_proj   = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)
        self.act_fn    = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))


# Test a single expert
expert = Expert(hidden_dim=64, intermediate_dim=256).to(device)
out = expert(x)
print(f"Expert input: {x.shape} -> output: {out.shape}")
print(f"Expert parameters: {sum(p.numel() for p in expert.parameters()):,}")

Expert input: torch.Size([2, 5, 64]) -> output: torch.Size([2, 5, 64])
Expert parameters: 49,152


In [5]:
# Step 3: Assemble the full MoE Layer

class MoELayer(nn.Module):
    """Mixture of Experts layer that replaces the standard FFN."""

    def __init__(self, hidden_dim: int, intermediate_dim: int,
                 num_experts: int = 8, top_k: int = 2):
        super().__init__()
        self.gate = TopKGating(hidden_dim, num_experts, top_k)
        self.experts = nn.ModuleList(
            [Expert(hidden_dim, intermediate_dim) for _ in range(num_experts)]
        )
        self.num_experts = num_experts
        self.top_k = top_k

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, hidden_dim)"""
        B, S, D = x.shape
        gate_probs, topk_idx = self.gate(x)  # (B,S,E), (B,S,k)

        # Compute each expert's output weighted by gate probability
        output = torch.zeros_like(x)
        for i, expert in enumerate(self.experts):
            # Mask: which tokens selected this expert
            expert_weight = gate_probs[:, :, i].unsqueeze(-1)  # (B, S, 1)
            # Only compute if some tokens selected this expert
            if expert_weight.sum() > 0:
                expert_out = expert(x)  # (B, S, D)
                output += expert_weight * expert_out

        return output


# Instantiate and test
moe = MoELayer(hidden_dim=64, intermediate_dim=256, num_experts=8, top_k=2).to(device)
moe_out = moe(x)

total_params = sum(p.numel() for p in moe.parameters())
single_expert_params = sum(p.numel() for p in moe.experts[0].parameters())
active_params = moe.top_k * single_expert_params + sum(p.numel() for p in moe.gate.parameters())

print(f"MoE output shape: {moe_out.shape}")
print(f"Total parameters:  {total_params:>10,}")
print(f"Active parameters: {active_params:>10,}  ({moe.top_k} experts + router)")
print(f"Sparsity ratio:    {active_params/total_params:.2%}")

MoE output shape: torch.Size([2, 5, 64])
Total parameters:     393,728
Active parameters:     98,816  (2 experts + router)
Sparsity ratio:    25.10%


## 3. Load Balancing in MoE Training

Without any constraint, the router tends to send most tokens to the same few "popular" experts, causing:
- Some experts are over-trained, others under-trained
- GPU load imbalance in distributed training

**Solution**: Add an auxiliary **load-balancing loss** that encourages uniform expert utilization:

$$\mathcal{L}_{\text{balance}} = \alpha \cdot N_E \cdot \sum_{i=1}^{N_E} f_i \cdot P_i$$

where $f_i$ is the fraction of tokens routed to expert $i$ and $P_i$ is the average router probability for expert $i$.

*Note: DeepSeek-V3 uses a different approach — shared experts with capacity constraints instead of auxiliary loss.*

In [6]:
def load_balancing_loss(gate_probs: torch.Tensor, topk_idx: torch.Tensor,
                        num_experts: int) -> torch.Tensor:
    """
    Compute Switch Transformer load-balancing loss.
    gate_probs: (B, S, E)
    topk_idx:   (B, S, k)
    """
    B, S, E = gate_probs.shape
    total_tokens = B * S

    # f_i: fraction of tokens routed to expert i
    expert_counts = torch.zeros(E, device=gate_probs.device)
    for i in range(E):
        expert_counts[i] = (topk_idx == i).float().sum()
    f = expert_counts / total_tokens

    # P_i: average probability assigned to expert i
    P = gate_probs.mean(dim=(0, 1))  # (E,)

    loss = E * (f * P).sum()
    return loss


# Demonstrate load balancing
gate, idx = gating(x)
lb_loss = load_balancing_loss(gate, idx, num_experts)

# Check expert distribution
print("=== Expert Load Distribution ===")
for i in range(num_experts):
    count = (idx == i).sum().item()
    avg_prob = gate[:, :, i].mean().item()
    print(f"  Expert {i}: {count:>3d} tokens, avg prob = {avg_prob:.4f}")
print(f"\nLoad-balancing loss: {lb_loss.item():.4f}")
print(f"Ideal (uniform): {1.0:.4f}")

=== Expert Load Distribution ===
  Expert 0:   3 tokens, avg prob = 0.1591
  Expert 1:   1 tokens, avg prob = 0.0525
  Expert 2:   1 tokens, avg prob = 0.0600
  Expert 3:   4 tokens, avg prob = 0.1349
  Expert 4:   2 tokens, avg prob = 0.0951
  Expert 5:   3 tokens, avg prob = 0.1277
  Expert 6:   2 tokens, avg prob = 0.1072
  Expert 7:   4 tokens, avg prob = 0.2635

Load-balancing loss: 2.3769
Ideal (uniform): 1.0000


## 4. Floating-Point Format Comparison

| Format | Sign | Exponent | Mantissa | Approx Range | Common Use |
|--------|:----:|:--------:|:--------:|:-------------|:-----------|
| FP32   | 1    | 8        | 23       | $\pm 3.4 \times 10^{38}$ | Full-precision training, critical evaluation |
| BF16   | 1    | 8        | 7        | $\pm 3.4 \times 10^{38}$ | Large-scale training (same range as FP32) |
| FP16   | 1    | 5        | 10       | $\pm 6.55 \times 10^{4}$ | Mixed-precision training & inference |
| FP8 E4M3 | 1 | 4        | 3        | $\pm 448$               | Latest GPU inference/training |
| FP8 E5M2 | 1 | 5        | 2        | $\pm 5.7 \times 10^{4}$ | Latest GPU training (wider range) |

**Trade-off**: More exponent bits → wider dynamic range; more mantissa bits → higher precision.

In [7]:
# Compare precision across formats
print("=== Floating-Point Format Properties ===")

formats = {
    "FP32": torch.float32,
    "FP16": torch.float16,
    "BF16": torch.bfloat16,
}

x_fp32 = torch.tensor([0.1, 1.0, 100.0, 65504.0, 1e-8], dtype=torch.float32)

print(f"{'Format':>6s} | {'Size (bits)':>11s} | {'eps':>12s} | {'max':>12s} | {'Representation of 0.1':>22s}")
print("-" * 75)
for name, dtype in formats.items():
    info = torch.finfo(dtype)
    val = torch.tensor(0.1, dtype=dtype)
    print(f"{name:>6s} | {info.bits:>11d} | {info.eps:>12.2e} | {info.max:>12.2e} | {val.item():.20f}")

# Demonstrate precision loss
print("\n=== Precision Loss When Casting ===")
large_val = torch.tensor(100000.0, dtype=torch.float32)
for name, dtype in formats.items():
    casted = large_val.to(dtype)
    # Check for overflow
    if casted.isinf():
        print(f"{name}: 100000.0 -> inf (OVERFLOW)")
    else:
        error = abs(casted.float().item() - large_val.item())
        print(f"{name}: 100000.0 -> {casted.float().item()}, error = {error}")

=== Floating-Point Format Properties ===
Format | Size (bits) |          eps |          max |  Representation of 0.1
---------------------------------------------------------------------------
  FP32 |          32 |     1.19e-07 |     3.40e+38 | 0.10000000149011611938
  FP16 |          16 |     9.77e-04 |     6.55e+04 | 0.09997558593750000000
  BF16 |          16 |     7.81e-03 |     3.39e+38 | 0.10009765625000000000

=== Precision Loss When Casting ===
FP32: 100000.0 -> 100000.0, error = 0.0
FP16: 100000.0 -> inf (OVERFLOW)
BF16: 100000.0 -> 99840.0, error = 160.0


## 5. Mixed Precision Training

Mixed precision combines FP32 (master weights, loss scaling) with FP16/BF16 (forward/backward computation) to:
- **Halve** memory usage
- **Double** throughput on Tensor Cores
- Maintain FP32-level training accuracy

Typical flow:
1. Forward pass in FP16/BF16
2. Loss computed in FP32 (with loss scaling for FP16)
3. Backward pass in FP16/BF16
4. Weight update in FP32 (master copy)

In [8]:
# Demonstrate mixed precision with a simple MLP
class SimpleMLP(nn.Module):
    def __init__(self, dim=256):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim * 4)
        self.fc2 = nn.Linear(dim * 4, dim)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))

dim = 256
model_fp32 = SimpleMLP(dim).to(device)

# Memory comparison
def model_memory(model, dtype=None):
    total = 0
    for p in model.parameters():
        if dtype:
            total += p.numel() * torch.tensor(0, dtype=dtype).element_size()
        else:
            total += p.numel() * p.element_size()
    return total

print("=== Memory Comparison ===")
for name, dtype in [("FP32", torch.float32), ("FP16", torch.float16), ("BF16", torch.bfloat16)]:
    mem = model_memory(model_fp32, dtype)
    print(f"{name}: {mem/1024:.1f} KB")

# Forward pass in different precisions
x_test = torch.randn(4, dim, device=device)

with torch.no_grad():
    out_fp32 = model_fp32(x_test)
    model_fp16 = SimpleMLP(dim).to(device).half()
    model_fp16.load_state_dict({k: v.half() for k, v in model_fp32.state_dict().items()})
    out_fp16 = model_fp16(x_test.half())

    error = (out_fp32 - out_fp16.float()).abs().mean().item()
    print(f"\nFP32 vs FP16 output mean absolute error: {error:.6f}")

=== Memory Comparison ===
FP32: 2053.0 KB
FP16: 1026.5 KB
BF16: 1026.5 KB

FP32 vs FP16 output mean absolute error: 0.000077


## 6. Quantization Basics

Quantization maps floating-point weights to a lower-bit integer representation:

$$x_q = \text{round}\left(\frac{x}{\text{scale}} + \text{zero\_point}\right)$$

$$\text{scale} = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}}, \qquad
  \text{zero\_point} = \text{round}\left(q_{\min} - \frac{x_{\min}}{\text{scale}}\right)$$

**Common approaches:**
- **Post-Training Quantization (PTQ)**: Quantize after training with a small calibration set
- **Quantization-Aware Training (QAT)**: Insert fake-quantize nodes during training
- **Weight-only INT8/INT4**: Quantize only weights; activations stay in FP16

In [9]:
def quantize_tensor(x: torch.Tensor, num_bits: int = 8):
    """Symmetric quantization of a float tensor to num_bits integers.

    Returns:
        x_q:   quantized integer tensor
        scale: float scale factor for dequantization
    """
    qmin = -(2 ** (num_bits - 1))
    qmax = 2 ** (num_bits - 1) - 1

    x_max = x.abs().max()
    scale = x_max / qmax

    x_q = torch.clamp(torch.round(x / scale), qmin, qmax).to(torch.int8)
    return x_q, scale


def dequantize_tensor(x_q: torch.Tensor, scale: float) -> torch.Tensor:
    """Dequantize: int tensor -> float tensor."""
    return x_q.float() * scale


# Demo: quantize model weights
weight = model_fp32.fc1.weight.detach().cpu()
print(f"Original weight: shape={weight.shape}, dtype={weight.dtype}")
print(f"  range: [{weight.min():.4f}, {weight.max():.4f}]")
print(f"  memory: {weight.numel() * weight.element_size()} bytes")

# INT8 quantization
w_q8, scale8 = quantize_tensor(weight, num_bits=8)
w_deq8 = dequantize_tensor(w_q8, scale8)
error8 = (weight - w_deq8).abs().mean().item()

print(f"\nINT8 quantized: dtype={w_q8.dtype}")
print(f"  memory: {w_q8.numel() * w_q8.element_size()} bytes ({weight.element_size()/w_q8.element_size():.0f}× smaller)")
print(f"  mean abs error: {error8:.6f}")

# INT4 quantization (simulated — stored in int8 container)
w_q4, scale4 = quantize_tensor(weight, num_bits=4)
w_deq4 = dequantize_tensor(w_q4, scale4)
error4 = (weight - w_deq4).abs().mean().item()

print(f"\nINT4 quantized:")
print(f"  effective memory: {weight.numel() * 0.5:.0f} bytes ({weight.element_size()/0.5:.0f}× smaller)")
print(f"  mean abs error: {error4:.6f}")

print(f"\n=== Summary ===")
print(f"FP32 error: 0.000000")
print(f"INT8 error: {error8:.6f}")
print(f"INT4 error: {error4:.6f}  (higher error, much smaller footprint)")

Original weight: shape=torch.Size([1024, 256]), dtype=torch.float32
  range: [-0.0625, 0.0625]
  memory: 1048576 bytes

INT8 quantized: dtype=torch.int8
  memory: 262144 bytes (4× smaller)
  mean abs error: 0.000123

INT4 quantized:
  effective memory: 131072 bytes (8× smaller)
  mean abs error: 0.002233

=== Summary ===
FP32 error: 0.000000
INT8 error: 0.000123
INT4 error: 0.002233  (higher error, much smaller footprint)


In [10]:
# Impact of quantization on model output
print("=== End-to-End Quantization Impact ===")

# Replace fc1 weight with dequantized INT8 weight
model_q8 = SimpleMLP(dim).to(device)
model_q8.load_state_dict(model_fp32.state_dict())

with torch.no_grad():
    # Quantize + dequantize fc1 weights
    w = model_q8.fc1.weight.data.cpu()
    w_q, s = quantize_tensor(w, 8)
    model_q8.fc1.weight.data = dequantize_tensor(w_q, s).to(device)

    # Similarly for fc2
    w2 = model_q8.fc2.weight.data.cpu()
    w2_q, s2 = quantize_tensor(w2, 8)
    model_q8.fc2.weight.data = dequantize_tensor(w2_q, s2).to(device)

    out_q8 = model_q8(x_test)
    out_orig = model_fp32(x_test)

output_error = (out_orig - out_q8).abs().mean().item()
relative_error = output_error / out_orig.abs().mean().item() * 100
print(f"Output mean abs error: {output_error:.6f}")
print(f"Relative error: {relative_error:.2f}%")

=== End-to-End Quantization Impact ===
Output mean abs error: 0.000817
Relative error: 0.57%


## Conclusions

### Technical Concepts Learned
- **MoE Architecture**: Sparse expert selection via Top-K gating, scaling capacity without scaling compute
- **Gating Network**: Router linear layer + Top-K + softmax; `scatter` for sparse gate construction
- **Load Balancing**: Auxiliary loss to prevent expert collapse during training
- **Floating-Point Formats**: FP32 (full precision) → BF16 (wide range, low precision) → FP16 (narrow range, higher precision) → FP8
- **Mixed Precision**: Using FP16/BF16 for compute with FP32 master weights for accuracy
- **Quantization**: Mapping float weights to INT8/INT4 for inference compression; trade-off between size and accuracy

### Experiment Further
- Modify `top_k` from 2 to 1 and observe the expert distribution change
- Compare per-channel vs per-tensor quantization accuracy
- Apply quantization to a real LLaMA model using `bitsandbytes` or `auto-gptq`
- Implement a capacity factor to limit maximum tokens per expert
- Benchmark MoE vs dense FFN with the same compute budget

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
